Анализ образовательных метрик
Добро пожаловать в Edtech! Продакт-менеджер Василий попросил вас проанализировать поведение студентов на курсах, предметах и экзаменах онлайн-школы и ответить на следующие вопросы:

1. Сколько студентов успешно сдали только один курс? (Успешная сдача — это зачёт по курсу на экзамене)

2. Найдите и отсортируйте id экзаменов в рамках курса по возрастанию уровня завершаемости*.

3. Выявите самые популярные предметы (ТОП-3) по количеству регистраций на них.

4. Выявите предметы с самым большим оттоком (ТОП-3).

5. Используя pandas, в период с начала 2013 по конец 2014 выявите семестр с самой низкой завершаемостью курсов.

6. Используя pandas, в период с начала 2013 по конец 2014 выявите семестр с самыми долгими средними сроками сдачи курсов.

7. Часто для качественного анализа аудитории используют подходы, основанные на сегментации. Используя python, постройте адаптированные RFM-кластеры студентов, чтобы качественно оценить свою аудиторию. В адаптированной кластеризации можно выбрать следующие метрики:

R — среднее время сдачи одного экзамена,

F — завершаемость курсов,

M — среднее количество баллов, получаемое за экзамен.

Для каждого RFM-сегмента постройте границы метрик recency, frequency и monetary для интерпретации этих кластеров. Описание подхода можно найти тут.

     7. 1. Чему равна минимальная граница по recency?

     7. 2. Чему равна максимальная граница по recency?

     7. 3. Чему равна минимальная граница по monetary?

     7. 4. Чему равна максимальная граница по monetary?

     7. 5. Сколько клиентов попадут в кластер 232?

              Используйте логику:

             a) по recency можно получить 2, если значение recency меньше либо равно,  медиане по recency. В остальных случаях — 1.

             b) по frequency можно получить 1, если значение frequency меньше 50. Можно получить 2, если значение по frequency меньше 100. Можно получить 3 в остальных случаях.

             c) по monetary можно получить 1, если значение monetary меньше 40. Можно получить 2, если значение по monetary меньше либо равно 80. Можно получить 3 в остальных случаях.

Для решения задачи проведите предварительное исследование данных и сформулируйте, что должно считаться курсом. Обосновать свой выбор вы можете с помощью фактов сдачи экзаменов, распределения студентов и уникального идентификатора курса.

*завершаемость = кол-во успешных экзаменов/кол-во всех попыток сдать экзамен * 100.

Файлы
1. Assessments.csv — файл содержит информацию об оценках в тесте. Обычно каждый предмет в семестре включает ряд тестов с оценками, за которыми следует заключительный экзаменационный тест (экзамен).

code_module — идентификационный код предмета.
code_presentation — семестр (Идентификационный код).
id_assessment — тест (Идентификационный номер ассессмента).
assessment_type — тип теста. Существуют три типа оценивания: оценка преподавателя (TMA), компьютерная оценка (СМА), экзамен по курсу (Exam).
date — информация об окончательной дате сдачи теста. Рассчитывается как количество дней с момента начала семестра. Дата начала семестра имеет номер 0 (ноль).
weight — вес теста в % в оценке за курс. Обычно экзамены рассматриваются отдельно и имеют вес 100%. Сумма всех остальных оценок составляет 100%.
2. Сourses.csv — файл содержит список предметов по семестрам.

code_module — предмет (идентификационный код).
code_presentation — семестр (идентификационный код).
module_presentation_length — продолжительность семестра в днях.
3. StudentAssessment.csv — файл содержит результаты тестов студентов. Если учащийся не отправляет работу на оценку, результат не записывается в таблицу.

id_assessment — тест (идентификационный номер).
id_student — идентификационный номер студента.
date_submitted — дата сдачи теста студентом, измеряемая как количество дней с начала семестра.
 is_banked — факт перезачета теста с прошлого семестра (иногда курсы перезачитывают студентам, вернувшимся из академического отпуска).
score — оценка учащегося в этом тесте. Диапазон составляет от 0 до 100. Оценка ниже 40 неудачная/неуспешная сдача теста.
4. StudentRegistration.csv — этот файл содержит информацию о времени, когда студент зарегистрировался для прохождения курса в семестре.

code_module — предмет (идентификационный код).
code_presentation — семестр (идентификационный код).
id_student — идентификационный номер студента.
date_registration — дата регистрации студента. Это количество дней, измеренное от начала семестра. Например, отрицательное значение -30 означает, что студент зарегистрировался на прохождение курса за 30 дней до его начала.
date_unregistration — дата отмены регистрации студента с предмета. У студентов, окончивших курс, это поле остается пустым.

In [172]:
import pandas as pd

In [173]:
student_assessment = pd.read_csv("data/StudentAssessment.csv ")
assessments = pd.read_csv("data/assessments.csv ")
courses = pd.read_csv("data/courses.csv ")
student_registration = pd.read_csv("data/studentRegistration.csv")

In [174]:
df = assessments.merge(student_assessment, on='id_assessment', how='inner')

Сколько студентов успешно сдали только один курс?

In [175]:
df_new = df.query("assessment_type == 'Exam' & score >= 40").groupby('id_student', as_index=False).size()
df_new.query('size == 1').nunique().id_student

np.int64(3802)

Найдите и отсортируйте id экзаменов в рамках курса по возрастанию уровня завершаемости:

In [176]:
df_exam = df.query("assessment_type == 'Exam'")

In [177]:
result = df_exam.groupby('id_assessment').apply(
    lambda x: pd.Series({
        'total_attempts': len(x),  # Общее количество попыток
        'successful_attempts': (x['score'] >= 40).sum(),  # Успешные попытки (оценка >= 40)
        'completion_rate': (x['score'] >= 40).mean()*100  # Процент завершённости
    })
).reset_index()

# Сортируем по уровню завершённости
result = result.sort_values(by='completion_rate', ascending=True)

# Форматируем результат
result = result[['id_assessment', 'completion_rate', 'total_attempts', 'successful_attempts']]

print(result)

   id_assessment  completion_rate  total_attempts  successful_attempts
2          25340        83.720930           602.0                504.0
1          24299        87.243151          1168.0               1019.0
5          25368        88.631579           950.0                842.0
0          24290        88.888889           747.0                664.0
3          25354        90.702479           968.0                878.0
4          25361        92.557252           524.0                485.0


C:\Users\user\AppData\Local\Temp\ipykernel_6628\352162901.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df_exam.groupby('id_assessment').apply(


Выявите самые популярные предметы (ТОП-3) по количеству регистраций на них:

In [178]:
popular_modules = student_registration.groupby('code_module').agg(
    total_registrations=('id_student', 'nunique')  # считаем уникальные регистрации
).reset_index()

# Сортируем по убыванию количества регистраций
popular_modules = popular_modules.sort_values(
    by='total_registrations', ascending=False
)

# Выводим топ-3
top_3_modules = popular_modules.head(3)

print("Топ-3 самых популярных предметов:")
print(top_3_modules[['code_module', 'total_registrations']])

Топ-3 самых популярных предметов:
  code_module  total_registrations
1         BBB                 7692
5         FFF                 7397
3         DDD                 5848


Выявите предметы с самым большим оттоком* (ТОП-3)

In [179]:
student_registration.groupby('code_module', as_index=False).apply(lambda x: pd.Series({
    'total_unregistration': (x['date_unregistration'] != None).sum()
    })
    ).sort_values(by='total_unregistration', ascending=False).head(3)

C:\Users\user\AppData\Local\Temp\ipykernel_6628\2638456022.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  student_registration.groupby('code_module', as_index=False).apply(lambda x: pd.Series({


,code_module,total_unregistration
1,BBB,7909
5,FFF,7762
3,DDD,6272


в период с начала 2013 по конец 2014 выявите семестр с самой низкой завершаемостью курсов:

In [180]:
popular_modules = student_registration.groupby('code_presentation').agg(
    total_registrations=('id_student', 'nunique')  # считаем уникальные регистрации
).reset_index()

# Сортируем по убыванию количества регистраций
popular_modules = popular_modules.sort_values(
    by='total_registrations', ascending=True
)
popular_modules.head(1)

,code_presentation,total_registrations
0,2013B,4679


в период с начала 2013 по конец 2014 выявите семестр с самыми долгими средними сроками сдачи курсов:

In [181]:
# Фильтруем данные по периоду 2013-2014
df_filtered = df[df['code_presentation'].str.contains('2013|2014')]

# Группируем по семестрам и считаем средний срок сдачи
avg_duration = df_filtered.groupby('code_presentation').agg(
    avg_duration=('date', 'mean')
).reset_index()

# Сортируем по убыванию среднего срока
avg_duration = avg_duration.sort_values(by='avg_duration', ascending=False)

print("Семестры с самыми долгими сроками сдачи:")
print(avg_duration)

Семестры с самыми долгими сроками сдачи:
  code_presentation  avg_duration
1             2013J    138.952181
3             2014J    130.220093
2             2014B    126.452541
0             2013B    123.764398


Часто для качественного анализа аудитории используют подходы, основанные на сегментации. Используя python, постройте адаптированные RFM-кластеры студентов, чтобы качественно оценить свою аудиторию. В адаптированной кластеризации можете выбрать следующие метрики:

R — среднее время сдачи одного экзамена,

F — завершаемость курсов,

M — среднее количество баллов, получаемое за экзамен.

Для каждого RFM-сегмента постройте границы метрик recency, frequency и monetary для интерпретации этих кластеров.


Чему равна минимальная граница по recency?

In [182]:
# Получение идентификаторов экзаменов
exams_id = df.query('assessment_type == "Exam"')['id_assessment']

# Фильтрация результатов экзаменов и агрегация
exams_results = df.query('id_assessment in @exams_id')

# Расчёт RFM метрик
rfm_exams = (
    exams_results
    .groupby('id_student', as_index=False)
    .agg(
        recency=('date_submitted', 'mean'),
        all_scores=('score', 'count')
    )
)

# Успешные экзамены
exams_success = exams_results.query('score >= 40')

# Расчёт успешных оценок
success_scores_by_students = (
    exams_success
    .groupby('id_student', as_index=False)
    .agg(success_scores=('score', 'count'))
)

# Объединение данных о RFM с успешными оценками
rfm_exams = rfm_exams.merge(success_scores_by_students, how='left', on='id_student').fillna(0)

# Расчёт частоты успешных оценок
rfm_exams['frequency'] = (100 * rfm_exams.success_scores / rfm_exams.all_scores).round(2)

# Расчёт средней оценки по всем экзаменам
all_scores_by_student = (
    exams_results
    .groupby('id_student', as_index=False)
    .agg(monetary=('score', 'mean'))
)

# Объединение с RFM данными
rfm_exams = rfm_exams.merge(all_scores_by_student, how='left', on='id_student').fillna(0)

# Выбор только необходимых столбцов
rfm_exams = rfm_exams[['id_student', 'recency', 'frequency', 'monetary']]

# Описание полученного DataFrame
print(rfm_exams.describe())

         id_student      recency    frequency     monetary
count  4.633000e+03  4633.000000  4633.000000  4633.000000
mean   7.256904e+05   238.462227    88.128642    65.117958
std    5.753498e+05     5.653378    32.114175    20.470561
min    2.369800e+04   229.000000     0.000000     0.000000
25%    5.011580e+05   234.000000   100.000000    50.000000
50%    5.884820e+05   241.000000   100.000000    66.000000
75%    6.463510e+05   243.000000   100.000000    82.000000
max    2.698251e+06   285.000000   100.000000   100.000000


In [183]:
rfm_exams

,id_student,recency,frequency,monetary
0,23698,243.0,100.0,80.0
1,24213,236.0,100.0,58.0
2,27116,243.0,100.0,96.0
3,28046,237.0,100.0,40.0
4,28787,243.0,100.0,44.0
...,...,...,...,...
4628,2694886,236.0,100.0,69.0
4629,2694933,230.0,100.0,73.0
4630,2695608,237.0,100.0,73.0
4631,2697181,230.0,100.0,80.0


Используя логику:

1) по recency можно получить 2, если значение recency меньше либо равно медиане по recency. В остальных случаях — 1.

2) по frequency можно получить 1, если значение frequency меньше 50. Можно получить 2, если значение по frequency меньше 100. Можно получить 3 в остальных случаях.

3) по monetary можно получить 1, если значение monetary меньше 40. Можно получить 2, если значение по monetary меньше либо равно 80. Можно получить 3 в остальных случаях.

Сколько клиентов попадут в кластер 232?

In [184]:
# Создаем кластеры
rfm_exams['recency'] = rfm_exams['recency'].apply(lambda x: 2 if x <= rfm_exams['recency'].describe()['50%'] else 1).astype(str)
rfm_exams['frequency'] = rfm_exams['frequency'].apply(lambda x: 1 if x < 50 else 2 if x < 100 else 3).astype(str)
rfm_exams['monetary'] = rfm_exams['monetary'].apply(lambda x: 1 if x < 40 else 2 if x <= 80 else 3).astype(str)

In [185]:

rfm_exams['cluster'] = rfm_exams['recency'] + rfm_exams['frequency'] + rfm_exams['monetary']


In [191]:
rfm_exams[rfm_exams.cluster == '232'].id_student.count()

np.int64(1555)